<a href="https://colab.research.google.com/github/Hakson-spec/Research_Paper_Code/blob/main/Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FusionGuard: CNN + LSTM + BERT Ensemble for Fake News Detection
# ------------------------------------------------------------
# Run this in Google Colab with a T4 GPU runtime.
#
# Before running:
# 1. Download "Fake.csv" and "True.csv" from the Kaggle dataset:
#    https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
# 2. Upload both files to the Colab session (left sidebar > Files > Upload).
#
# WHAT'S NEW IN THIS VERSION:
# - Prints standalone accuracy for CNN, LSTM, and BERT individually, not just
#   the final ensemble — no more separate cell needed afterward.
# - Saves cnn_probs.npy, lstm_probs.npy, bert_probs.npy, and y_test.npy to
#   /content/ the moment they're computed. Download these immediately after
#   the run finishes (folder icon > right-click each file > Download) as a
#   backup — if your session disconnects afterward, you can re-upload them
#   instead of retraining from scratch.
# - Generates fig2_branch_comparison.png automatically.
# ============================================================

# %% [1] Install dependencies
# Colab already has tensorflow, torch, scikit-learn, pandas, matplotlib, seaborn,
# and nltk pre-installed. It's just missing "transformers". The BERT branch below
# uses transformers' PyTorch classes (not the TensorFlow ones), so there's no
# tf-keras version conflict to worry about here.
# !pip install -q transformers

# %% [2] Imports
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, f1_score
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout,
    Bidirectional, LSTM
)
from tensorflow.keras.models import Model

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizerFast, BertForSequenceClassification

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUT_DIR = "/content"  # everything gets saved here — download these before disconnecting

# %% [3] Load data ------------------------------------------------
DATA_DIR = "."  # change if your CSVs are elsewhere
fake = pd.read_csv(f"{DATA_DIR}/Fake.csv")
real = pd.read_csv(f"{DATA_DIR}/True.csv")

fake["label"] = 1  # 1 = fake
real["label"] = 0  # 0 = real

df = pd.concat([fake, real], ignore_index=True)
df["content"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
df = df[["content", "label"]].dropna().reset_index(drop=True)

print("Class distribution:")
print(df["label"].value_counts())

# %% [4] Clean text (with data-leakage fix) --------------------------
LEAK_WORDS = {"reuters", "via", "com", "pic", "twitter", "featured", "getty"}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"^\s*[a-z][a-z\s,\.]*\(reuters\)\s*-\s*", " ", text)
    text = re.sub(r"pic\.?twitter\.?com\S*", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [w for w in text.split() if w not in STOPWORDS and w not in LEAK_WORDS and len(w) > 2]
    return " ".join(tokens)

df["clean"] = df["content"].apply(clean_text)

# %% [5] Train/test split -------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.20, stratify=df["label"], random_state=SEED
)
print(f"Train size: {len(X_train_text)} | Test size: {len(X_test_text)}")
np.save(f"{OUT_DIR}/y_test.npy", y_test.values)

# OPTIONAL: set this to a number (e.g. 8000) for a fast first-pass test run of the
# whole pipeline. Set back to None before generating the final numbers for your paper.
SUBSAMPLE_TRAIN = None
if SUBSAMPLE_TRAIN is not None:
    X_train_text = X_train_text.sample(SUBSAMPLE_TRAIN, random_state=SEED)
    y_train = y_train.loc[X_train_text.index]
    print(f"Subsampled training set down to {len(X_train_text)} rows for a faster test run.")

# ============================================================
# BRANCH 1: CNN
# ============================================================
MAX_VOCAB = 20000
MAX_LEN = 300

tokenizer_kf = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer_kf.fit_on_texts(X_train_text)

X_train_seq = pad_sequences(tokenizer_kf.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding="post")
X_test_seq = pad_sequences(tokenizer_kf.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding="post")

def build_cnn():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(MAX_VOCAB, 128)(inp)
    x = Conv1D(128, 5, activation="relu")(x)
    x = GlobalMaxPooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

cnn_model = build_cnn()
cnn_model.fit(X_train_seq, y_train, validation_split=0.1, epochs=5, batch_size=64, verbose=1)
cnn_probs = cnn_model.predict(X_test_seq).ravel()
np.save(f"{OUT_DIR}/cnn_probs.npy", cnn_probs)
print(f"CNN standalone accuracy: {accuracy_score(y_test, (cnn_probs >= 0.5).astype(int)) * 100:.2f}%")

# ============================================================
# BRANCH 2: Bi-LSTM
# ============================================================
def build_lstm():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(MAX_VOCAB, 128)(inp)
    x = Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3))(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

lstm_model = build_lstm()
lstm_model.fit(X_train_seq, y_train, validation_split=0.1, epochs=5, batch_size=64, verbose=1)
lstm_probs = lstm_model.predict(X_test_seq).ravel()
np.save(f"{OUT_DIR}/lstm_probs.npy", lstm_probs)
print(f"LSTM standalone accuracy: {accuracy_score(y_test, (lstm_probs >= 0.5).astype(int)) * 100:.2f}%")

# ============================================================
# BRANCH 3: Fine-tuned BERT (PyTorch backend)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("BERT branch using device:", device)
if device.type != "cuda":
    print("=" * 70)
    print("WARNING: No GPU detected — BERT training will take HOURS on CPU.")
    print("Go to Runtime > Change runtime type > select T4 GPU > Save,")
    print("then re-run this cell. Do not continue on CPU.")
    print("=" * 70)

BERT_MAX_LEN = 128
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

print("Tokenizing training set...")
train_enc = bert_tokenizer(
    list(X_train_text), truncation=True, padding="max_length",
    max_length=BERT_MAX_LEN, return_tensors="pt",
)
print("Tokenizing test set...")
test_enc = bert_tokenizer(
    list(X_test_text), truncation=True, padding="max_length",
    max_length=BERT_MAX_LEN, return_tensors="pt",
)
train_labels = torch.tensor(y_train.values, dtype=torch.long)
test_labels = torch.tensor(y_test.values, dtype=torch.long)

class EncodedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = EncodedDataset(train_enc, train_labels)
test_dataset = EncodedDataset(test_enc, test_labels)

BATCH_SIZE = 32 if device.type == "cuda" else 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

bert_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
optimizer = AdamW(bert_model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

BERT_EPOCHS = 2
bert_model.train()
for epoch in range(BERT_EPOCHS):
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch + 1} step {step}/{len(train_loader)} — loss {loss.item():.4f}")
    print(f"BERT epoch {epoch + 1}/{BERT_EPOCHS} — avg training loss: {total_loss / len(train_loader):.4f}")

bert_model.eval()
bert_probs_batches = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits.float(), dim=1)[:, 1]
        bert_probs_batches.append(probs.cpu().numpy())
bert_probs = np.concatenate(bert_probs_batches)
np.save(f"{OUT_DIR}/bert_probs.npy", bert_probs)
print(f"BERT standalone accuracy: {accuracy_score(y_test, (bert_probs >= 0.5).astype(int)) * 100:.2f}%")
print(f"\nSaved cnn_probs.npy, lstm_probs.npy, bert_probs.npy, y_test.npy to {OUT_DIR}")
print("DOWNLOAD THESE NOW (folder icon > right-click each > Download) as a backup.")

# ============================================================
# ENSEMBLE FUSION (soft voting) — matches Section III.D
# ============================================================
final_probs = (cnn_probs + lstm_probs + bert_probs) / 3.0
y_pred = (final_probs >= 0.5).astype(int)

cnn_acc = accuracy_score(y_test, (cnn_probs >= 0.5).astype(int)) * 100
lstm_acc = accuracy_score(y_test, (lstm_probs >= 0.5).astype(int)) * 100
bert_acc = accuracy_score(y_test, (bert_probs >= 0.5).astype(int)) * 100
ensemble_acc = accuracy_score(y_test, y_pred) * 100

print("\n--- TABLE 1 VALUES (all measured on the same test set) ---")
print(f"CNN (standalone): {cnn_acc:.2f}%")
print(f"LSTM (standalone): {lstm_acc:.2f}%")
print(f"BERT (standalone): {bert_acc:.2f}%")
print(f"FusionGuard (Ensemble): {ensemble_acc:.2f}%")

# Fig. 2 — bar chart comparing accuracy across all four (for the Methodology doc)
branch_results = {
    "CNN": cnn_acc, "LSTM": lstm_acc, "BERT": bert_acc, "FusionGuard\n(Ensemble)": ensemble_acc,
}
plt.figure(figsize=(6, 4))
bars = plt.bar(branch_results.keys(), branch_results.values(), color=["#4C72B0", "#55A868", "#DD8452", "#C44E52"])
plt.ylabel("Accuracy (%)"); plt.title("Accuracy Comparison Across Branches"); plt.ylim(0, 105)
for b, v in zip(bars, branch_results.values()):
    plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.2f}%", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_branch_comparison.png", dpi=200); plt.close()

# ============================================================
# METRICS ---> fills Section V and Table II
# ============================================================
report = classification_report(y_test, y_pred, target_names=["Real", "Fake"], output_dict=True)
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"Sensitivity (Fake): {sensitivity*100:.2f}%")
print(f"Specificity (Real): {specificity*100:.2f}%")

# FIGURE 1: Confusion Matrix
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Normalized)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_confusion_matrix.png", dpi=200); plt.close()

# FIGURE 2 (ROC): ROC Curve
fpr, tpr, _ = roc_curve(y_test, final_probs)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"FusionGuard (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_roc_curve.png", dpi=200); plt.close()

# FIGURE 3: F1-Score per class
classes = ["Real", "Fake"]
f1_scores = [report["Real"]["f1-score"], report["Fake"]["f1-score"]]
plt.figure(figsize=(5, 4))
bars = plt.bar(classes, f1_scores, color=["#4C72B0", "#DD8452"])
plt.ylim(0, 1); plt.ylabel("F1 Score"); plt.title("F1 Score per Class")
for b, v in zip(bars, f1_scores):
    plt.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_f1_per_class.png", dpi=200); plt.close()

# FIGURE 4: Precision / Recall / F1 grouped bar chart
metrics_df = pd.DataFrame({
    "Precision": [report["Real"]["precision"], report["Fake"]["precision"]],
    "Recall": [report["Real"]["recall"], report["Fake"]["recall"]],
    "F1-Score": [report["Real"]["f1-score"], report["Fake"]["f1-score"]],
}, index=classes)
metrics_df.plot(kind="bar", figsize=(6, 4), ylim=(0, 1))
plt.title("Evaluation Metrics per Class"); plt.ylabel("Score"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_metrics_bar.png", dpi=200); plt.close()

# FIGURE 5: Class distribution
plt.figure(figsize=(5, 4))
df["label"].map({0: "Real", 1: "Fake"}).value_counts().plot(kind="bar", color=["#55A868", "#C44E52"])
plt.title("Class Distribution (Full Dataset)"); plt.ylabel("Article Count")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig5_class_distribution.png", dpi=200); plt.close()

# FIGURE 6: Feature importance (TF-IDF + chi-square)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(X_train_text)
chi2_scores, _ = chi2(X_tfidf, y_train)
feat_names = np.array(tfidf.get_feature_names_out())
top_idx = np.argsort(chi2_scores)[-20:]
plt.figure(figsize=(6, 6))
plt.barh(feat_names[top_idx], chi2_scores[top_idx], color="#4C72B0")
plt.xlabel("Chi-square score"); plt.title("Top 20 Features by Chi-square Importance")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig6_feature_importance.png", dpi=200); plt.close()

# ============================================================
# TABLE III: Baseline comparison models (TF-IDF based)
# ============================================================
tfidf_full = TfidfVectorizer(max_features=20000)
Xtr = tfidf_full.fit_transform(X_train_text)
Xte = tfidf_full.transform(X_test_text)

baselines = {
    "Naive Bayes": MultinomialNB(),
    "SVM (TF-IDF)": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}

print("\n--- Table III values ---")
for name, clf in baselines.items():
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    print(f"{name}: {accuracy_score(y_test, pred)*100:.2f}%")
print(f"FusionGuard (Proposed): {ensemble_acc:.2f}%")

print("\nAll figures + .npy backups saved to", OUT_DIR)
print("Download everything now via the folder icon before this session ends.")



Class distribution:
label
1    23481
0    21417
Name: count, dtype: int64
Train size: 35918 | Test size: 8980
Epoch 1/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9353 - loss: 0.1560 - val_accuracy: 0.9736 - val_loss: 0.0698
Epoch 2/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9937 - loss: 0.0212 - val_accuracy: 0.9869 - val_loss: 0.0474
Epoch 3/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9995 - loss: 0.0027 - val_accuracy: 0.9866 - val_loss: 0.0577
Epoch 4/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9998 - loss: 8.6612e-04 - val_accuracy: 0.9855 - val_loss: 0.0598
Epoch 5/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 1.0000 - loss: 2.6479e-04 - val_accuracy: 0.9858 - val_loss: 0.0633
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
CNN standalone accuracy: 98.81%
Epoch 1/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 984s 2s/step - accuracy: 0.9419 - loss: 0.1505 - val_accuracy: 0.9811 - val_loss: 0.0553
Epoch 2/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 970s 

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing training set...
Tokenizing test set...


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_1993/67

  epoch 1 step 0/1123 — loss 0.7202
  epoch 1 step 200/1123 — loss 0.2059
  epoch 1 step 400/1123 — loss 0.0020
  epoch 1 step 600/1123 — loss 0.0102
  epoch 1 step 800/1123 — loss 0.0043
  epoch 1 step 1000/1123 — loss 0.0253
BERT epoch 1/2 — avg training loss: 0.0641
  epoch 2 step 0/1123 — loss 0.0020
  epoch 2 step 200/1123 — loss 0.0012
  epoch 2 step 400/1123 — loss 0.0005
  epoch 2 step 600/1123 — loss 0.0344
  epoch 2 step 800/1123 — loss 0.0073
  epoch 2 step 1000/1123 — loss 0.0025
BERT epoch 2/2 — avg training loss: 0.0158


/tmp/ipykernel_1993/674561344.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


BERT standalone accuracy: 99.32%

Saved cnn_probs.npy, lstm_probs.npy, bert_probs.npy, y_test.npy to /content
DOWNLOAD THESE NOW (folder icon > right-click each > Download) as a backup.

--- TABLE 1 VALUES (all measured on the same test set) ---
CNN (standalone): 98.81%
LSTM (standalone): 98.26%
BERT (standalone): 99.32%
FusionGuard (Ensemble): 99.62%
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      4284
        Fake       1.00      1.00      1.00      4696

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

Sensitivity (Fake): 99.57%
Specificity (Real): 99.67%

--- Table III values ---
Naive Bayes: 93.23%
SVM (TF-IDF): 98.95%
Random Forest: 98.30%
Logistic Regression: 97.92%
FusionGuard (Proposed): 99.62%

All figures + .npy backups saved to /content
Download everything now via the folder icon before this session ends.


In [ ]:
   !ls -la

total 20
drwxr-xr-x 1 root root 4096 Aug 20 11:18 .
drwxr-xr-x 1 root root 4096 Aug 20 11:29 ..
drwxr-xr-x 4 root root 4096 Aug 10 13:31 .config
drwx------ 5 root root 4096 Aug 20 11:18 drive
drwxr-xr-x 1 root root 4096 Aug 10 13:31 sample_data


In [ ]:
!cp /Fake.csv /True.csv /cnn_probs.npy /lstm_probs.npy /bert_probs.npy /y_test.npy /content/
!ls -la /content

total 113856
drwxr-xr-x 1 root root     4096 Aug 20 11:36 .
drwxr-xr-x 1 root root     4096 Aug 20 11:29 ..
-rw-r--r-- 1 root root    36048 Aug 20 11:36 bert_probs.npy
-rw-r--r-- 1 root root    36048 Aug 20 11:36 cnn_probs.npy
drwxr-xr-x 4 root root     4096 Aug 10 13:31 .config
drwx------ 5 root root     4096 Aug 20 11:18 drive
-rw-r--r-- 1 root root 62789876 Aug 20 11:36 Fake.csv
-rw-r--r-- 1 root root    36048 Aug 20 11:36 lstm_probs.npy
drwxr-xr-x 1 root root     4096 Aug 10 13:31 sample_data
-rw-r--r-- 1 root root 53582940 Aug 20 11:36 True.csv
-rw-r--r-- 1 root root    71968 Aug 20 11:36 y_test.npy


In [ ]:
# ============================================================
# FusionGuard — Finish Up (no retraining, no GPU needed)
# ------------------------------------------------------------
# Uses the cnn_probs.npy / lstm_probs.npy / bert_probs.npy / y_test.npy files
# you already have saved locally. Re-derives the cleaned text (deterministic,
# same random seed as before, so the split matches exactly) to train the
# lightweight Table III baseline models and build the feature-importance /
# class-distribution charts — nothing here needs a GPU.
#
# Before running:
# 1. CPU runtime is fine (Runtime > Change runtime type > CPU).
# 2. Upload: Fake.csv, True.csv, cnn_probs.npy, lstm_probs.npy, bert_probs.npy,
#    y_test.npy — all six files, via the folder icon.
# ============================================================

# %% [1] Imports
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, accuracy_score
)

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
SEED = 42
np.random.seed(SEED)
OUT_DIR = "/content"

# %% [2] Re-derive the exact same cleaned data & split as before -----------
DATA_DIR = "."
fake = pd.read_csv(f"{DATA_DIR}/Fake.csv")
real = pd.read_csv(f"{DATA_DIR}/True.csv")
fake["label"] = 1
real["label"] = 0
df = pd.concat([fake, real], ignore_index=True)
df["content"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
df = df[["content", "label"]].dropna().reset_index(drop=True)

LEAK_WORDS = {"reuters", "via", "com", "pic", "twitter", "featured", "getty"}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"^\s*[a-z][a-z\s,\.]*\(reuters\)\s*-\s*", " ", text)
    text = re.sub(r"pic\.?twitter\.?com\S*", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [w for w in text.split() if w not in STOPWORDS and w not in LEAK_WORDS and len(w) > 2]
    return " ".join(tokens)

df["clean"] = df["content"].apply(clean_text)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.20, stratify=df["label"], random_state=SEED
)

# %% [3] Load your saved probabilities & sanity-check they match -----------
cnn_probs = np.load(f"{DATA_DIR}/cnn_probs.npy")
lstm_probs = np.load(f"{DATA_DIR}/lstm_probs.npy")
bert_probs = np.load(f"{DATA_DIR}/bert_probs.npy")
y_test_saved = np.load(f"{DATA_DIR}/y_test.npy")

assert np.array_equal(y_test.values, y_test_saved), \
    "y_test doesn't match the saved file — the split may differ. Stop and let me know."
print("Sanity check passed — reproduced split matches your saved y_test exactly.")

final_probs = (cnn_probs + lstm_probs + bert_probs) / 3.0
y_pred = (final_probs >= 0.5).astype(int)

cnn_acc = accuracy_score(y_test, (cnn_probs >= 0.5).astype(int)) * 100
lstm_acc = accuracy_score(y_test, (lstm_probs >= 0.5).astype(int)) * 100
bert_acc = accuracy_score(y_test, (bert_probs >= 0.5).astype(int)) * 100
ensemble_acc = accuracy_score(y_test, y_pred) * 100

print("\n--- TABLE 1 VALUES ---")
print(f"CNN (standalone): {cnn_acc:.2f}%")
print(f"LSTM (standalone): {lstm_acc:.2f}%")
print(f"BERT (standalone): {bert_acc:.2f}%")
print(f"FusionGuard (Ensemble): {ensemble_acc:.2f}%")

# Fig. 2 — branch comparison bar chart
branch_results = {"CNN": cnn_acc, "LSTM": lstm_acc, "BERT": bert_acc, "FusionGuard\n(Ensemble)": ensemble_acc}
plt.figure(figsize=(6, 4))
bars = plt.bar(branch_results.keys(), branch_results.values(), color=["#4C72B0", "#55A868", "#DD8452", "#C44E52"])
plt.ylabel("Accuracy (%)"); plt.title("Accuracy Comparison Across Branches"); plt.ylim(0, 105)
for b, v in zip(bars, branch_results.values()):
    plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.2f}%", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_branch_comparison.png", dpi=200); plt.close()

# %% [4] Ensemble metrics + Figures 1,2(ROC),3,4,5,6 ------------------------
report = classification_report(y_test, y_pred, target_names=["Real", "Fake"], output_dict=True)
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Sensitivity (Fake): {tp/(tp+fn)*100:.2f}%")
print(f"Specificity (Real): {tn/(tn+fp)*100:.2f}%")

cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Normalized)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_confusion_matrix.png", dpi=200); plt.close()

fpr, tpr, _ = roc_curve(y_test, final_probs)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"FusionGuard (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_roc_curve.png", dpi=200); plt.close()

classes = ["Real", "Fake"]
f1_scores = [report["Real"]["f1-score"], report["Fake"]["f1-score"]]
plt.figure(figsize=(5, 4))
bars = plt.bar(classes, f1_scores, color=["#4C72B0", "#DD8452"])
plt.ylim(0, 1); plt.ylabel("F1 Score"); plt.title("F1 Score per Class")
for b, v in zip(bars, f1_scores):
    plt.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_f1_per_class.png", dpi=200); plt.close()

metrics_df = pd.DataFrame({
    "Precision": [report["Real"]["precision"], report["Fake"]["precision"]],
    "Recall": [report["Real"]["recall"], report["Fake"]["recall"]],
    "F1-Score": [report["Real"]["f1-score"], report["Fake"]["f1-score"]],
}, index=classes)
metrics_df.plot(kind="bar", figsize=(6, 4), ylim=(0, 1))
plt.title("Evaluation Metrics per Class"); plt.ylabel("Score"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_metrics_bar.png", dpi=200); plt.close()

plt.figure(figsize=(5, 4))
df["label"].map({0: "Real", 1: "Fake"}).value_counts().plot(kind="bar", color=["#55A868", "#C44E52"])
plt.title("Class Distribution (Full Dataset)"); plt.ylabel("Article Count")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig5_class_distribution.png", dpi=200); plt.close()

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(X_train_text)
chi2_scores, _ = chi2(X_tfidf, y_train)
feat_names = np.array(tfidf.get_feature_names_out())
top_idx = np.argsort(chi2_scores)[-20:]
plt.figure(figsize=(6, 6))
plt.barh(feat_names[top_idx], chi2_scores[top_idx], color="#4C72B0")
plt.xlabel("Chi-square score"); plt.title("Top 20 Features by Chi-square Importance")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig6_feature_importance.png", dpi=200); plt.close()

# %% [5] Table III baselines (fast — classical ML only) --------------------
tfidf_full = TfidfVectorizer(max_features=20000)
Xtr = tfidf_full.fit_transform(X_train_text)
Xte = tfidf_full.transform(X_test_text)

baselines = {
    "Naive Bayes": MultinomialNB(),
    "SVM (TF-IDF)": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}
print("\n--- Table III values ---")
for name, clf in baselines.items():
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    print(f"{name}: {accuracy_score(y_test, pred)*100:.2f}%")
print(f"FusionGuard (Proposed): {ensemble_acc:.2f}%")

print(f"\nAll figures saved to {OUT_DIR} — download them via the folder icon.")


Sanity check passed — reproduced split matches your saved y_test exactly.

--- TABLE 1 VALUES ---
CNN (standalone): 98.81%
LSTM (standalone): 98.26%
BERT (standalone): 99.32%
FusionGuard (Ensemble): 99.62%
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      4284
        Fake       1.00      1.00      1.00      4696

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

Sensitivity (Fake): 99.57%
Specificity (Real): 99.67%

--- Table III values ---
Naive Bayes: 93.23%
SVM (TF-IDF): 98.95%
Random Forest: 98.30%
Logistic Regression: 97.92%
FusionGuard (Proposed): 99.62%

All figures saved to /content — download them via the folder icon.
